# Phase 0.4 — Stockfish in Colab Test

This notebook tests Stockfish evaluation performance on Google Colab.
It supports both **CPU runtimes** and **GPU runtimes**.

### Objectives
1. Install a modern Stockfish binary with NNUE support.
2. Install `python-chess`.
3. Load 100 diverse chess positions (30 openings, 40 middlegames, 30 endgames).
4. Benchmark Single-PV analysis at fixed depth 18.
5. Benchmark MultiPV=3 analysis at fixed depth 18 (required for Phase 2.2).
6. Test CPU multithreading scaling.
7. Project computation time for 10,000 and 50,000 positions.
8. Output benchmark numbers for `docs/decisions/compute.md`.


In [ ]:
# Step 1: Install Stockfish universal binary and python-chess
import os
import stat
import subprocess
import sys
from pathlib import Path

# Install python-chess
!pip install -q python-chess

# Download official Stockfish universal Linux binary
SF_DIR = Path("/content/stockfish_bin")
SF_DIR.mkdir(parents=True, exist_ok=True)
SF_BIN = SF_DIR / "stockfish"

if not SF_BIN.exists():
    print("Downloading official Stockfish universal Linux binary...")
    tar_url = "https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-linux-x86-64-universal.tar.gz"
    tar_path = SF_DIR / "sf.tar.gz"
    try:
        !curl -L -s "{tar_url}" -o "{tar_path}"
        !tar -xzf "{tar_path}" -C "{SF_DIR}"
        # Find extracted binary
        for p in SF_DIR.glob("**/stockfish*"):
            if p.is_file() and not p.name.endswith(".tar.gz") and not p.name.endswith(".txt"):
                p.chmod(p.stat().st_mode | stat.S_IEXEC)
                SF_BIN = p
                break
        print(f"Stockfish installed at: {SF_BIN}")
    except Exception as exc:
        print(f"Download failed: {exc}. Trying apt-get fallback...")
        !apt-get update -qq && apt-get install -y -qq stockfish
        SF_BIN = Path("/usr/games/stockfish")

# Verify Stockfish execution and identity
res = subprocess.run([str(SF_BIN), "uci"], capture_output=True, text=True, timeout=5)
for line in res.stdout.splitlines():
    if "id name" in line:
        print(f"Engine Active: {line.strip()}")
        break


In [ ]:
# Step 2: Load 100 diverse chess positions
# 30 openings, 40 middlegames, 30 endgames
import os

try:
    from positions import BENCHMARK_POSITIONS
    POSITIONS = BENCHMARK_POSITIONS
except ImportError:
    # If uploaded as a single notebook to Colab, use embedded positions
# 100 Diverse, realistic chess positions (30 openings, 40 middlegames, 30 endgames)
# Shared between benchmark_stockfish.py and phase0_4_stockfish_test.ipynb

BENCHMARK_POSITIONS = [
    # --- 30 OPENING POSITIONS ---
    "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1",
    "rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    "rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    "r1bqkbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 3",
    "r1bqk1nr/pppp1ppp/2n5/2b1p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    "r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    "r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3",
    "r1bqk2r/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQ1RK1 b kq - 5 4",
    "rnbqkb1r/pp2pppp/3p1n2/8/3NP3/8/PPP2PPP/RNBQKB1R w KQkq - 1 5",
    "rnbqk2r/pp2bppp/3p1n2/4p3/3NP3/2N5/PPP1BPPP/R1BQK2R w KQkq - 2 7",
    "rnbqkb1r/pp3ppp/3p1n2/2p1p3/2P1P3/2N2N2/PP1P1PPP/R1BQKB1R w KQkq - 0 5",
    "rnbqkbnr/ppp1pppp/8/3p4/2PP4/8/PP2PPPP/RNBQKBNR b KQkq - 0 2",
    "rnbqkb1r/ppp2ppp/4pn2/3p4/2PP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 2 4",
    "rnbqkb1r/ppp1pp1p/5np1/3p4/2PP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 0 4",
    "rnbqkb1r/pppppp1p/5np1/8/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 2",
    "rnbq1rk1/ppp1ppbp/3p1np1/8/2PPP3/2N2N2/PP2BPPP/R1BQK2R b KQ - 3 6",
    "rnbqkb1r/pp1ppppp/5n2/2p5/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3",
    "r1bqkbnr/pp1ppppp/2n5/2p5/2P5/2N5/PP1PPPPP/R1BQKBNR w KQkq - 2 3",
    "rnbqkb1r/pp1p1ppp/4pn2/2p5/2PP4/5N2/PP2PPPP/RNBQKB1R w KQkq - 0 4",
    "r1bqk2r/pppp1ppp/2n2n2/4p3/1bB1P3/2N2N2/PPPP1PPP/R1BQK2R w KQkq - 6 5",
    "rnbqkb1r/pp2pppp/2p2n2/3p4/2PP4/5N2/PP2PPPP/RNBQKB1R w KQkq - 0 4",
    "rnbqkb1r/ppp2ppp/4pn2/8/2pP4/4PN2/PP3PPP/RNBQKB1R w KQkq - 0 5",
    "r1bqkbnr/pppp1ppp/2n5/4p3/3PP3/5N2/PPP2PPP/RNBQKB1R b KQkq - 0 3",
    "rnbqkbnr/pp2pppp/2p5/3p4/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 0 3",
    "rnbqkbnr/ppp1pppp/8/3p4/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    "rnbqkbnr/pppp1ppp/8/4p3/4PP2/8/PPPP2PP/RNBQKBNR b KQkq - 0 2",
    "rnbqkbnr/pppp1ppp/8/4p3/2B1P3/8/PPPP1PPP/RNBQK1NR b KQkq - 1 2",
    "rnbqkbnr/ppp1pppp/8/3p4/3P4/8/PPP1PPPP/RNBQKBNR w KQkq - 0 2",
    "rnbqkb1r/pppp1ppp/4pn2/8/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3",
    "rnbqkbnr/pppppp1p/8/6p1/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",

    # --- 40 MIDDLEGAME POSITIONS ---
    "r1bq1rk1/pp2ppbp/2np1np1/8/2BNP3/2N1BP2/PPP3PP/R2QK2R w KQ - 1 9",
    "r1b2rk1/pp1n1ppp/2p1pn2/q2p2B1/2PP4/P1P1PN2/2Q2PPP/R3KB1R w KQ - 3 10",
    "r2q1rk1/1b2bppp/p2p1n2/1p2p3/3pP3/PB1P3P/1PP1NPP1/R1BQ1RK1 w - - 2 13",
    "r1b1qrk1/ppp2ppp/2n1pn2/3p4/2PP4/2N1PN2/PP2BPPP/R1BQ1RK1 w - - 4 8",
    "2r2rk1/1pq1bppp/p2p1n2/4p3/P3P3/1PN1BP2/1P4PP/2RQ1RK1 w - - 3 16",
    "r2qr1k1/pb1nbppp/1p1ppn2/8/2PP4/1PN2NP1/PB3PBP/R2Q1RK1 w - - 1 12",
    "r1b2rk1/1pq1bppp/p1nppn2/8/3NPP2/2N1B3/PPP1B1PP/R2Q1RK1 w - - 2 11",
    "r4rk1/1pp1qppp/p1np1n2/4p1B1/2B1P1b1/P1NP1N2/1PP2PPP/R2Q1RK1 w - - 2 10",
    "r2q1rk1/pb2bppp/1pn1pn2/2pp4/2PP4/1PN1PN2/PB2BPPP/R2Q1RK1 w - - 0 10",
    "2r1r1k1/pp3ppp/2n1p3/q2p4/3P4/P1PB3P/1P1Q1PP1/R3R1K1 w - - 1 18",
    "r1b2rk1/ppqn1ppp/2pbpn2/3p4/2PP4/1P2PN2/PB1N1PPP/R2QKB1R w KQ - 3 9",
    "r1bq1rk1/1pp2ppp/p1np1n2/2b1p3/2B1P3/2NP1N2/PPPB1PPP/R2Q1RK1 w - - 0 8",
    "r3k2r/pbqnbppp/1p1ppn2/2p5/2PPPP2/2NB1N2/PP2Q1PP/R1B2RK1 w kq - 2 10",
    "r2q1rk1/1p2bppp/p1npbn2/4p3/4P3/1NN1BP2/PPP1B1PP/R2Q1RK1 w - - 3 11",
    "3r1rk1/ppq2ppp/2n1pn2/8/1bPP4/2N1BB2/PP3PPP/R2Q1RK1 w - - 1 13",
    "r1b2rk1/pp1nqppp/4p3/2ppP3/3P4/2PB1N2/PP1Q1PPP/R3K2R w KQ - 0 11",
    "r1bq1rk1/pp2ppbp/1n1p1np1/2pP4/2P1P3/2N2N2/PP2BPPP/R1BQ1RK1 w - - 1 9",
    "2rq1rk1/pb1nbppp/1p2pn2/2pp4/2PP4/1PN1PN2/PB2BPPP/2RQ1RK1 w - - 0 11",
    "r1bqr1k1/pp1nbppp/2p2n2/3p2B1/3P4/2NBPN2/PP3PPP/R2QK2R w KQ - 3 10",
    "r2q1rk1/1bp1bppp/p1n1pn2/1p6/3P4/1BN1PN2/PP3PPP/R1BQ1RK1 w - - 2 11",
    "r1bq1rk1/1p2bppp/p1nppn2/8/3NP3/2N1BP2/PPP1B1PP/R2Q1RK1 w - - 0 10",
    "r1b2rk1/2q1bppp/p2p1n2/npp1p3/3PP3/2P2N1P/PPB2PP1/RNBQR1K1 w - - 1 12",
    "r1b2rk1/pp3ppp/2n1pn2/q1bp4/2PP4/2N2NP1/PP2PPBP/R1BQ1RK1 w - - 3 9",
    "r2q1rk1/1p1bbppp/p1nppn2/8/P2NPP2/2N1B3/1PP1B1PP/R2Q1RK1 w - - 1 11",
    "r1b1r1k1/pp3pbp/1qp2np1/4p3/2P5/2N1PBBP/PP3PP1/R2Q1RK1 w - - 2 14",
    "r2q1rk1/pp1n1ppp/2pbpn2/3p4/2PP4/1PN1PN2/PB2QPPP/R4RK1 w - - 1 11",
    "r1bq1rk1/1pp1bppp/p1n1pn2/3p4/2PP4/2NBPN2/PP3PPP/R1BQ1RK1 w - - 0 8",
    "r2qr1k1/1pp1nppp/pb1p1n2/4p3/B3P1b1/2PP1N2/PP1N1PPP/R1BQR1K1 w - - 4 12",
    "r1b2rk1/pp3ppp/2n1p3/q1bpP3/3N1P2/2N1B3/PPP3PP/R2Q1RK1 w - - 1 12",
    "r1b2rk1/pp1nqppp/2n1p3/3pP3/3P4/3B1N2/PP1N1PPP/R2QK2R w KQ - 1 11",
    "2r2rk1/1p1b1ppp/pq1ppb2/8/2PNPP2/1PN5/P2Q2PP/3R1RK1 w - - 1 18",
    "r2q1rk1/1b1n1ppp/p3pn2/1pb5/8/1PN1PN2/PB2BPPP/R2Q1RK1 w - - 0 12",
    "r1b1kb1r/pp3ppp/2n1p3/q2pP3/3P4/P1N1PN2/1P4PP/R2QKB1R b KQkq - 0 10",
    "r1bq1rk1/1p2bppp/p1nppn2/6B1/3NP3/2N5/PPP1BPPP/R2Q1RK1 w - - 0 10",
    "r2q1rk1/1p1bbppp/p1n1pn2/3p4/2PP4/2NB1N2/PP3PPP/R1BQR1K1 w - - 2 11",
    "r2q1rk1/pb1nbppp/1p1ppn2/8/3PP3/2NB1N2/PP2QPPP/R1B2RK1 w - - 4 11",
    "r1b2rk1/pp3pbp/1qnp1np1/4p3/2P1P3/2N2N1P/PP2BPP1/R1BQ1RK1 w - - 1 11",
    "r1bq1rk1/ppp2ppp/2n1pn2/3p4/1bPP4/2N1PN2/PP1B1PPP/R2QKB1R w KQ - 3 7",
    "r1bq1rk1/pp2ppbp/2np1np1/8/3NP3/2N1BP2/PPP3PP/2KR1B1R w - - 1 10",
    "r2q1rk1/pb2bppp/1pn1pn2/2pp4/2PP4/2N1PN2/PP1BBPPP/R2Q1RK1 w - - 2 10",

    # --- 30 ENDGAME POSITIONS ---
    "8/8/4k3/8/8/4K3/4P3/8 w - - 0 1",
    "8/5pk1/7p/4P3/5PK1/8/8/8 w - - 1 45",
    "8/8/8/4k3/8/2P1K3/8/8 w - - 0 1",
    "8/8/1p6/p1k5/P7/1PK5/8/8 w - - 4 50",
    "8/2R5/8/8/4k3/r7/3K4/8 w - - 0 60",
    "8/8/8/8/k7/2R5/r7/2K5 w - - 0 70",
    "4r3/6k1/6p1/5p1p/7P/4RKP1/8/8 b - - 0 55",
    "8/8/5k2/R7/4K3/8/8/5r2 w - - 0 65",
    "8/8/1p1k4/1P3p2/3K1P2/8/8/8 w - - 2 55",
    "8/5k2/3p4/p1pP1p1p/P1P2P1P/8/5K2/8 w - - 0 60",
    "8/8/2b5/8/8/4K3/5B2/4k3 b - - 0 70",
    "8/8/4k3/8/8/2B1K3/4P3/8 w - - 0 80",
    "8/8/3n4/8/3k4/2N5/3K4/8 w - - 0 85",
    "8/8/8/8/4n3/2k5/3N4/2K5 w - - 0 90",
    "8/8/8/4k3/8/3K4/4Q3/7q b - - 0 75",
    "8/8/8/4k3/8/3K4/8/7q w - - 0 80",
    "8/8/1k6/8/8/1K6/2R5/r7 w - - 0 65",
    "8/8/8/1p6/1k6/8/1PK5/8 w - - 0 52",
    "8/4k3/8/3P4/4K3/8/8/8 w - - 0 1",
    "8/8/8/8/8/3k4/3p4/3K4 b - - 0 1",
    "2r5/8/4k3/R7/8/4K3/8/8 w - - 0 58",
    "8/3k4/3P4/3K4/8/8/8/8 w - - 0 1",
    "8/8/8/8/2p1k3/2P5/4K3/8 w - - 0 50",
    "8/8/2k5/p1p5/P1P5/1K6/8/8 w - - 0 55",
    "8/8/8/4b3/8/2k1K3/8/5B2 w - - 0 65",
    "8/8/8/8/3k4/3P4/3K4/8 w - - 0 1",
    "6k1/5p2/6p1/7p/7P/6P1/5PK1/8 w - - 0 45",
    "8/8/8/2R5/1r6/4k3/8/2K5 w - - 0 62",
    "8/5k2/5P2/4K3/8/8/8/8 w - - 0 1",
    "8/8/8/8/4k3/8/4K3/4R3 w - - 0 1"
]

    POSITIONS = BENCHMARK_POSITIONS

print(f"Loaded {len(POSITIONS)} benchmark positions (30 openings, 40 middlegames, 30 endgames).")


In [ ]:
# Step 3: Run Single-PV benchmark (Depth 18, 1 Thread)
import time
import chess
import chess.engine

print("Starting Single-PV Benchmark (Depth 18, Threads=1, 100 positions)...")

engine = chess.engine.SimpleEngine.popen_uci(str(SF_BIN))
engine.configure({"Threads": 1, "Hash": 128})

single_times = []
single_nodes = 0
t0 = time.time()

for idx, fen in enumerate(POSITIONS):
    board = chess.Board(fen)
    p_start = time.time()
    info = engine.analyse(board, chess.engine.Limit(depth=18), multipv=1)
    single_times.append(time.time() - p_start)
    if isinstance(info, list):
        single_nodes += sum(item.get("nodes", 0) for item in info)
    else:
        single_nodes += info.get("nodes", 0)
    if (idx + 1) % 25 == 0:
        print(f"Completed {idx + 1}/100 positions (current avg: {sum(single_times)/len(single_times):.3f}s/pos)")

single_total = time.time() - t0
engine.quit()

single_avg = sum(single_times) / len(single_times)
single_median = sorted(single_times)[len(single_times) // 2]
single_10k_hrs = (single_avg * 10_000) / 3600.0
single_50k_hrs = (single_avg * 50_000) / 3600.0

print("\n--- Single-PV Results ---")
print(f"Total Time (100 pos): {single_total:.2f} s")
print(f"Average per pos:      {single_avg:.3f} s")
print(f"Median per pos:       {single_median:.3f} s")
print(f"Projected 10k pos:    {single_10k_hrs:.2f} hours")
print(f"Projected 50k pos:    {single_50k_hrs:.2f} hours")


In [ ]:
# Step 4: Run MultiPV=3 benchmark (Depth 18, 1 Thread) — Required for Phase 2.2
print("Starting MultiPV=3 Benchmark (Depth 18, Threads=1, 100 positions)...")

engine = chess.engine.SimpleEngine.popen_uci(str(SF_BIN))
engine.configure({"Threads": 1, "Hash": 128})

multi_times = []
multi_nodes = 0
t0 = time.time()

for idx, fen in enumerate(POSITIONS):
    board = chess.Board(fen)
    p_start = time.time()
    info = engine.analyse(board, chess.engine.Limit(depth=18), multipv=3)
    multi_times.append(time.time() - p_start)
    if isinstance(info, list):
        multi_nodes += max((item.get("nodes", 0) for item in info), default=0)
    else:
        multi_nodes += info.get("nodes", 0)
    if (idx + 1) % 25 == 0:
        print(f"Completed {idx + 1}/100 positions (current avg: {sum(multi_times)/len(multi_times):.3f}s/pos)")

multi_total = time.time() - t0
engine.quit()

multi_avg = sum(multi_times) / len(multi_times)
multi_median = sorted(multi_times)[len(multi_times) // 2]
multi_10k_hrs = (multi_avg * 10_000) / 3600.0
multi_50k_hrs = (multi_avg * 50_000) / 3600.0

print("\n--- MultiPV=3 Results ---")
print(f"Total Time (100 pos): {multi_total:.2f} s")
print(f"Average per pos:      {multi_avg:.3f} s")
print(f"Median per pos:       {multi_median:.3f} s")
print(f"Projected 10k pos:    {multi_10k_hrs:.2f} hours")
print(f"Projected 50k pos:    {multi_50k_hrs:.2f} hours")


In [ ]:
# Step 5: Test Multi-threading Scaling (2 Threads vs 1 Thread on 20 positions)
print("Evaluating thread scaling on 20 test positions...")
test_subset = POSITIONS[:20]

engine_2t = chess.engine.SimpleEngine.popen_uci(str(SF_BIN))
engine_2t.configure({"Threads": 2, "Hash": 128})

t0 = time.time()
for fen in test_subset:
    b = chess.Board(fen)
    engine_2t.analyse(b, chess.engine.Limit(depth=18), multipv=3)
t_2threads = time.time() - t0
engine_2t.quit()

avg_2t = t_2threads / 20.0
speedup = (multi_avg / avg_2t) if avg_2t > 0 else 1.0

print(f"MultiPV=3 with 2 Threads: {avg_2t:.3f} s/pos (Speedup: {speedup:.2f}x vs 1 Thread)")
print(f"Projected 10k positions with 2 Threads: {(avg_2t * 10_000) / 3600.0:.2f} hours")


In [ ]:
# Step 6: Print Markdown summary table for docs/decisions/compute.md
summary_md = f"""
## Phase 0.4 Stockfish Benchmark Results (Colab)

| Evaluation Mode | Depth | Threads | 100 Positions Time | Avg Time / Position | Projected 10k Positions | Projected 50k Positions |
|---|---|---|---|---|---|---|
| **Single-PV** | 18 | 1 | {single_total:.2f} s | {single_avg:.3f} s | {single_10k_hrs:.2f} hours | {single_50k_hrs:.2f} hours |
| **MultiPV=3** | 18 | 1 | {multi_total:.2f} s | {multi_avg:.3f} s | {multi_10k_hrs:.2f} hours | {multi_50k_hrs:.2f} hours |
| **MultiPV=3** | 18 | 2 | {t_2threads * 5:.2f} s (est) | {avg_2t:.3f} s | {(avg_2t * 10000) / 3600.0:.2f} hours | {(avg_2t * 50000) / 3600.0:.2f} hours |

### Key Takeaways for Data Pipeline (Phase 2.2)
- Colab CPU runtimes are suitable for generating Stockfish evaluations.
- MultiPV=3 increases computation time per position by approximately {multi_avg / single_avg:.2f}x compared to Single-PV.
- Using 2 CPU threads reduces pipeline run time by ~{(1.0 - 1.0/speedup)*100:.0f}%.
- All position evaluations must be cached by FEN on Google Drive in Parquet format to prevent redundant computations.
"""

print(summary_md)
